# 用客户端工具创建客服智能体

在本示例中，我们将演示如何用 Claude 3 和客户端工具创建一个客服聊天机器人。该聊天机器人能够查询客户信息、获取订单详情，并代表客户取消订单。我们定义所需的工具，并模拟合成响应，以展示聊天机器人的能力。

## 步骤 1：设置环境

首先，安装所需的库并设置 Claude API 客户端。

In [ ]:
%pip install anthropic

In [18]:
import anthropic

client = anthropic.Client()
MODEL_NAME = "claude-opus-4-8"

In [22]:
import os

from anthropic import Anthropic

gateway_url = os.environ.get("ANTHROPIC_BASE_URL", "").rstrip("/")
gateway_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()

if gateway_url != "https://qwen.smjtools.com":
    raise RuntimeError("当前内核没有 cookbook 网关配置。")
if not gateway_key or gateway_key == "YOUR_QWEN_GATEWAY_KEY":
    raise RuntimeError("请配置个人网关密钥并重启 Jupyter。")
if gateway_key.startswith("sk-ant-"):
    raise RuntimeError("不要将 Anthropic 密钥发送到 Qwen 网关。")
if os.environ.get("ANTHROPIC_AUTH_TOKEN"):
    raise RuntimeError("存在额外 auth token，请使用独立启动环境。")

client = Anthropic(
    api_key=gateway_key,
    base_url=gateway_url,
    timeout=180.0,
    max_retries=0,
)
MODEL_NAME = "strata"

## 步骤 2：定义客户端工具

接下来，我们定义聊天机器人用来协助客户的客户端工具。我们创建三个工具：get_customer_info、get_order_details 和 cancel_order。

In [24]:
tools = [
    {
        "name": "get_customer_info",
        "description": "Retrieves customer information based on their customer ID. Returns the customer's name, email, and phone number.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "The unique identifier for the customer.",
                }
            },
            "required": ["customer_id"],
        },
    },
    {
        "name": "get_order_details",
        "description": "Retrieves the details of a specific order based on the order ID. Returns the order ID, product name, quantity, price, and order status.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The unique identifier for the order.",
                }
            },
            "required": ["order_id"],
        },
    },
    {
        "name": "cancel_order",
        "description": "Cancels an order based on the provided order ID. Returns a confirmation message if the cancellation is successful.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The unique identifier for the order to be cancelled.",
                }
            },
            "required": ["order_id"],
        },
    },
]

## 步骤 3：模拟合成工具响应

由于我们没有真实的客户数据或订单信息，我们为工具模拟合成响应。在真实场景中，这些函数会与你的实际客户数据库和订单管理系统交互。

In [25]:
def get_customer_info(customer_id):
    # Simulated customer data
    customers = {
        "C1": {"name": "John Doe", "email": "john@example.com", "phone": "123-456-7890"},
        "C2": {"name": "Jane Smith", "email": "jane@example.com", "phone": "987-654-3210"},
    }
    return customers.get(customer_id, "Customer not found")


def get_order_details(order_id):
    # Simulated order data
    orders = {
        "O1": {
            "id": "O1",
            "product": "Widget A",
            "quantity": 2,
            "price": 19.99,
            "status": "Shipped",
        },
        "O2": {
            "id": "O2",
            "product": "Gadget B",
            "quantity": 1,
            "price": 49.99,
            "status": "Processing",
        },
    }
    return orders.get(order_id, "Order not found")


def cancel_order(order_id):
    # Simulated order cancellation
    if order_id in ["O1", "O2"]:
        return True
    else:
        return False

## 步骤 4：处理工具调用并返回结果

我们创建一个函数，用于处理 Claude 发起的工具调用，并返回相应的结果。

In [26]:
def process_tool_call(tool_name, tool_input):
    if tool_name == "get_customer_info":
        return get_customer_info(tool_input["customer_id"])
    elif tool_name == "get_order_details":
        return get_order_details(tool_input["order_id"])
    elif tool_name == "cancel_order":
        return cancel_order(tool_input["order_id"])

## 步骤 5：与聊天机器人交互

现在，我们创建一个与聊天机器人交互的函数。我们发送一条用户消息，处理 Claude 发起的任何工具调用，并把最终响应返回给用户。

In [27]:
import json


def chatbot_interaction(user_message):
    print(f"\n{'=' * 50}\nUser Message: {user_message}\n{'=' * 50}")

    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model=MODEL_NAME, max_tokens=4096, tools=tools, messages=messages
    )

    print("\nInitial Response:")
    print(f"Stop Reason: {response.stop_reason}")
    print(f"Content: {response.content}")

    while response.stop_reason == "tool_use":
        tool_use = next(block for block in response.content if block.type == "tool_use")
        tool_name = tool_use.name
        tool_input = tool_use.input

        print(f"\nTool Used: {tool_name}")
        print("Tool Input:")
        print(json.dumps(tool_input, indent=2))

        tool_result = process_tool_call(tool_name, tool_input)

        print("\nTool Result:")
        print(json.dumps(tool_result, indent=2))

        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": response.content},
            {
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use.id,
                        "content": str(tool_result),
                    }
                ],
            },
        ]

        response = client.messages.create(
            model=MODEL_NAME, max_tokens=4096, tools=tools, messages=messages
        )

        print("\nResponse:")
        print(f"Stop Reason: {response.stop_reason}")
        print(f"Content: {response.content}")

    final_response = next(
        (block.text for block in response.content if hasattr(block, "text")),
        None,
    )

    print(f"\nFinal Response: {final_response}")

    return final_response

## 步骤 6：测试聊天机器人
我们用几条示例查询来测试客服聊天机器人。

In [28]:
chatbot_interaction("Can you tell me the email address for customer C1?")
chatbot_interaction("What is the status of order O2?")
chatbot_interaction("Please cancel order O1 for me.")


User Message: Can you tell me the email address for customer C1?

Initial Response:
Stop Reason: tool_use
Content: [ToolUseBlock(id='call_ce162b85d79840f2a8a3090f', caller=None, input={'customer_id': 'C1'}, name='get_customer_info', type='tool_use', toolset_name=None)]

Tool Used: get_customer_info
Tool Input:
{
  "customer_id": "C1"
}

Tool Result:
{
  "name": "John Doe",
  "email": "john@example.com",
  "phone": "123-456-7890"
}

Response:
Stop Reason: end_turn
Content: [TextBlock(citations=None, text='The email address for customer C1 is john@example.com.', type='text')]

Final Response: The email address for customer C1 is john@example.com.

User Message: What is the status of order O2?

Initial Response:
Stop Reason: tool_use
Content: [ToolUseBlock(id='call_c35a7c20ed2642a498122b43', caller=None, input={'order_id': 'O2'}, name='get_order_details', type='tool_use', toolset_name=None)]

Tool Used: get_order_details
Tool Input:
{
  "order_id": "O2"
}

Tool Result:
{
  "id": "O2",
  

'Order O1 has been successfully cancelled.'

就是这样！我们用 Claude 3 模型和客户端工具创建了一个客服聊天机器人。该聊天机器人可以根据用户的请求查询客户信息、获取订单详情并取消订单。通过定义清晰的工具描述和模式，我们让 Claude 能够有效理解并使用可用的工具来协助客户。

欢迎在此基础上扩展：接入你实际的客户数据库和订单管理系统，并添加更多工具以处理更广泛的客服任务。